In [0]:
# ============================================================
# PARAMÈTRES
# ============================================================
dbutils.widgets.text("catalog",        "banking")
dbutils.widgets.text("schema_bronze",  "bronze")
dbutils.widgets.text("schema_silver",  "silver")

catalog       = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")

# Tables
source_table     = f"{catalog}.{schema_bronze}.bronze_customers"
target_silver    = f"{catalog}.{schema_silver}.silver_customers"
monitoring_table = f"{catalog}.{schema_bronze}.execution_monitoring"

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import row_number
from delta.tables import DeltaTable

from pyspark.sql.window import Window

target_table = f"{catalog}.{schema_bronze}.bronze_customers"
target_silver = f"{catalog}.{schema_silver}.silver_customers"

# Get watermark first
try:
      date_max=spark.sql(f""" SELECT COALESCE(MAX(updated_at), '1900-01-01')
        FROM {target_silver}
                   """).collect()[0][0]
except Exception as e:
      print(f"Table n'existe pas encore : {e}")
      date_max = '1900-01-01'

print(f'Watermark: {date_max}')

# Read and filter by watermark EARLY to reduce data volume
df=spark.read.table(target_table)
df = df.filter(F.col("updated_at") > date_max)


# Gérer les nulls
df=df.filter(F.col('customer_id').isNotNull())

# Gérer les doublons
window=Window.partitionBy('customer_id').orderBy(F.col('updated_at').desc())
df=df.withColumn('row_number',row_number().over(window))
df=df.filter(F.col('row_number')==1).drop('row_number')

# Calculer l'âge
df=df.withColumn('age', F.floor(F.datediff(F.current_timestamp(), F.col('date_of_birth')) / 365).cast('int'))

# Ajouter timestamp de chargement silver
df=df.withColumn('silver_loaded_at', F.current_timestamp())

print(f'Lignes après transformations: {df.count()}')

table_exists = spark.catalog.tableExists(target_silver)

if not table_exists:
    print("Premier run — création de la table...")
    df.write.format("delta").mode("overwrite").option("delta.feature.allowColumnDefaults", "supported").saveAsTable(target_silver)
    print("✅ Table silver_customers créée")
else :
      print('merge en cours')
      DeltaTable.forName(spark, target_silver).alias("t") \
        .merge(df.alias("s"), "t.customer_id = s.customer_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
      print('merge terminé')